# Notebook 04 — Learned Residual Correction

**Repo:** `residual-phase-lock`  
**Notebook:** `04_learned_residual_correction.ipynb`

## Claim

> Residual structure can be learned and used to reduce drift.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
```

Notebook 03 used a known constraint. Notebook 04 removes that advantage: the correction is learned from the residual itself.

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Works in GitHub Actions from repo root.
# Works in Colab if notebook is opened from the GitHub repo.
# Fallback helps if Colab starts inside notebooks/.
if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(45)

NOTEBOOK_ID = "04"
NOTEBOOK_SLUG = "learned_residual_correction"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Generate signal with hidden residual structure

The signal contains:

1. a linear trend,
2. a smooth nonlinear residual structure,
3. small random noise.

The baseline model will only learn the linear trend.  
The residual correction model will learn the missing structure from the residual.

In [ ]:
n = 700
x = np.linspace(0, 12, n)

trend = 0.65 * x + 1.25
hidden_structure = (
    0.75 * np.sin(1.7 * x)
    + 0.28 * np.cos(3.4 * x)
    + 0.018 * (x - 6) ** 2
)
noise = np.random.normal(0, 0.18, size=n)

y = trend + hidden_structure + noise
X = x.reshape(-1, 1)

# Ordered split: learn residual on first part, validate correction on held-out later region.
split = int(0.65 * n)

X_train, X_test = X[:split], X[split:]
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]
hidden_train, hidden_test = hidden_structure[:split], hidden_structure[split:]

data_df = pd.DataFrame({
    "x": x,
    "observed_y": y,
    "trend": trend,
    "hidden_structure": hidden_structure,
    "noise": noise,
    "split": ["train" if i < split else "test" for i in range(n)],
})

# Raw synthetic data is useful for inspection but optional to commit.
exp.save_csv(data_df, "signal_data")

data_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.3)
plt.plot(x, trend, label="true trend", linewidth=2)
plt.axvline(x[split], linestyle="--", linewidth=1, label="train/test split")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Signal with hidden residual structure")
plt.legend()
plt.tight_layout()
exp.save_fig("signal_with_hidden_structure")
plt.show()

## 3. Baseline model: fit only the trend

The baseline is intentionally incomplete.  
It captures the main trend while leaving structured residuals.

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

y_train_hat = baseline_model.predict(X_train)
y_test_hat = baseline_model.predict(X_test)

train_residual = y_train - y_train_hat
test_residual = y_test - y_test_hat

baseline_train_rmse = float(np.sqrt(mean_squared_error(y_train, y_train_hat)))
baseline_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_hat)))
baseline_train_r2 = float(r2_score(y_train, y_train_hat))
baseline_test_r2 = float(r2_score(y_test, y_test_hat))

print(f"Baseline train RMSE: {baseline_train_rmse:.4f}")
print(f"Baseline test RMSE:  {baseline_test_rmse:.4f}")
print(f"Baseline train R²:   {baseline_train_r2:.4f}")
print(f"Baseline test R²:    {baseline_test_r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_train, y_train, label="train signal", linewidth=1.2)
plt.plot(x_test, y_test, label="test signal", linewidth=1.2)
plt.plot(x_train, y_train_hat, label="baseline fit train", linewidth=2)
plt.plot(x_test, y_test_hat, label="baseline fit test", linewidth=2)
plt.axvline(x[split], linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Baseline trend fit")
plt.legend()
plt.tight_layout()
exp.save_fig("baseline_trend_fit")
plt.show()

## 4. Learn residual structure

We train a residual model on the baseline residual:

```text
residual_model(x) ≈ y - baseline_model(x)
```

This is the learned correction step:

```text
baseline output + learned residual → corrected output
```

In [ ]:
# Polynomial features provide a simple smooth residual model.
# This is deliberately lightweight and interpretable.
residual_model = make_pipeline(
    PolynomialFeatures(degree=9, include_bias=False),
    LinearRegression(),
)

residual_model.fit(X_train, train_residual)

train_residual_hat = residual_model.predict(X_train)
test_residual_hat = residual_model.predict(X_test)

y_train_corrected = y_train_hat + train_residual_hat
y_test_corrected = y_test_hat + test_residual_hat

corrected_train_rmse = float(np.sqrt(mean_squared_error(y_train, y_train_corrected)))
corrected_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_corrected)))
corrected_train_r2 = float(r2_score(y_train, y_train_corrected))
corrected_test_r2 = float(r2_score(y_test, y_test_corrected))

train_rmse_reduction = float(baseline_train_rmse - corrected_train_rmse)
test_rmse_reduction = float(baseline_test_rmse - corrected_test_rmse)
relative_test_rmse_reduction = float(test_rmse_reduction / baseline_test_rmse)

residual_corr_train = float(np.corrcoef(train_residual, train_residual_hat)[0, 1])
residual_corr_test = float(np.corrcoef(test_residual, test_residual_hat)[0, 1])

print(f"Corrected train RMSE: {corrected_train_rmse:.4f}")
print(f"Corrected test RMSE:  {corrected_test_rmse:.4f}")
print(f"Corrected train R²:   {corrected_train_r2:.4f}")
print(f"Corrected test R²:    {corrected_test_r2:.4f}")
print(f"Relative test RMSE reduction: {relative_test_rmse_reduction:.4f}")
print(f"Residual corr train: {residual_corr_train:.4f}")
print(f"Residual corr test:  {residual_corr_test:.4f}")

In [ ]:
residual_learning_df = pd.DataFrame({
    "x": x,
    "split": ["train" if i < split else "test" for i in range(n)],
    "true_hidden_structure": hidden_structure,
    "baseline_residual": np.concatenate([train_residual, test_residual]),
    "learned_residual": np.concatenate([train_residual_hat, test_residual_hat]),
})

exp.save_csv(residual_learning_df, "residual_learning_table")

plt.figure(figsize=(10, 5))
plt.plot(x, hidden_structure, label="true hidden structure", linewidth=2)
plt.plot(x_train, train_residual_hat, label="learned residual train", linewidth=1.6)
plt.plot(x_test, test_residual_hat, label="learned residual test", linewidth=1.6)
plt.axvline(x[split], linestyle="--", linewidth=1, label="train/test split")
plt.xlabel("x")
plt.ylabel("residual component")
plt.title("Learned residual correction tracks hidden structure")
plt.legend()
plt.tight_layout()
exp.save_fig("learned_residual_structure")
plt.show()

## 5. Compare baseline and corrected outputs

The corrected output adds the learned residual back to the baseline trend prediction.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_test, y_test, label="test signal", linewidth=1.5)
plt.plot(x_test, y_test_hat, label="baseline prediction", linewidth=2)
plt.plot(x_test, y_test_corrected, label="corrected prediction", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Held-out correction: baseline vs learned residual correction")
plt.legend()
plt.tight_layout()
exp.save_fig("heldout_correction")
plt.show()

In [ ]:
conditions = pd.DataFrame({
    "condition": ["baseline_train", "corrected_train", "baseline_test", "corrected_test"],
    "rmse": [
        baseline_train_rmse,
        corrected_train_rmse,
        baseline_test_rmse,
        corrected_test_rmse,
    ],
    "r2": [
        baseline_train_r2,
        corrected_train_r2,
        baseline_test_r2,
        corrected_test_r2,
    ],
})

exp.save_csv(conditions, "baseline_vs_corrected")

conditions

In [ ]:
plt.figure(figsize=(8, 4))
labels = ["Train baseline", "Train corrected", "Test baseline", "Test corrected"]
values = [
    baseline_train_rmse,
    corrected_train_rmse,
    baseline_test_rmse,
    corrected_test_rmse,
]
plt.bar(labels, values)
plt.ylabel("RMSE")
plt.title("RMSE before and after learned residual correction")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
exp.save_fig("rmse_before_after")
plt.show()

## 6. Residual after correction

A successful learned correction should reduce the structured residual remaining after prediction.

In [ ]:
corrected_train_residual = y_train - y_train_corrected
corrected_test_residual = y_test - y_test_corrected

residual_compare_df = pd.DataFrame({
    "x": x,
    "split": ["train" if i < split else "test" for i in range(n)],
    "baseline_residual": np.concatenate([train_residual, test_residual]),
    "corrected_residual": np.concatenate([corrected_train_residual, corrected_test_residual]),
})

exp.save_csv(residual_compare_df, "residuals_before_after")

plt.figure(figsize=(10, 5))
plt.plot(x_train, train_residual, label="baseline residual train", linewidth=1.2)
plt.plot(x_test, test_residual, label="baseline residual test", linewidth=1.2)
plt.plot(x_train, corrected_train_residual, label="corrected residual train", linewidth=1.2)
plt.plot(x_test, corrected_test_residual, label="corrected residual test", linewidth=1.2)
plt.axvline(x[split], linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("residual")
plt.title("Residuals before and after learned correction")
plt.legend()
plt.tight_layout()
exp.save_fig("residuals_before_after")
plt.show()

## 7. Residual spectrum before and after

We compare spectral concentration before and after correction.  
A lower dominant-mode concentration after correction indicates that structured residual drift has been reduced.

In [ ]:
def spectral_structure_score(residual, x_axis, top_k=3):
    centered = residual - residual.mean()
    freqs = np.fft.rfftfreq(len(centered), d=(x_axis[1] - x_axis[0]))
    spectrum = np.abs(np.fft.rfft(centered))
    top_idx = np.argsort(spectrum)[-top_k:][::-1]
    total_energy = np.sum(spectrum**2)
    dominant_energy = np.sum(spectrum[top_idx]**2)
    score = float(dominant_energy / total_energy)
    return freqs, spectrum, top_idx, score

freq_base, spec_base, top_base, score_base = spectral_structure_score(test_residual, x_test)
freq_corr, spec_corr, top_corr, score_corr = spectral_structure_score(corrected_test_residual, x_test)

spectrum_compare = pd.DataFrame({
    "frequency": freq_base,
    "baseline_spectrum": spec_base,
    "corrected_spectrum": spec_corr,
})

exp.save_csv(spectrum_compare, "test_residual_spectrum_before_after")

dominant_modes = pd.DataFrame({
    "rank": np.arange(1, 4),
    "baseline_frequency": freq_base[top_base],
    "baseline_amplitude": spec_base[top_base],
    "corrected_frequency": freq_corr[top_corr],
    "corrected_amplitude": spec_corr[top_corr],
})

exp.save_csv(dominant_modes, "dominant_modes_before_after")

print(f"Baseline test residual structure score:  {score_base:.4f}")
print(f"Corrected test residual structure score: {score_corr:.4f}")

dominant_modes

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(freq_base, spec_base, label="baseline residual spectrum", linewidth=1.5)
plt.plot(freq_corr, spec_corr, label="corrected residual spectrum", linewidth=1.5)
plt.xlim(0, 2.5)
plt.xlabel("frequency")
plt.ylabel("amplitude")
plt.title("Held-out residual spectrum before and after correction")
plt.legend()
plt.tight_layout()
exp.save_fig("residual_spectrum_before_after")
plt.show()

## 8. Summary outputs

The summary table is saved into `results/04_summary.csv` and `results/04_summary.json`.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_train_rmse",
        "corrected_train_rmse",
        "baseline_test_rmse",
        "corrected_test_rmse",
        "test_rmse_reduction",
        "relative_test_rmse_reduction",
        "baseline_test_r2",
        "corrected_test_r2",
        "residual_corr_train",
        "residual_corr_test",
        "baseline_test_residual_structure_score",
        "corrected_test_residual_structure_score",
        "structure_score_reduction",
    ],
    "value": [
        baseline_train_rmse,
        corrected_train_rmse,
        baseline_test_rmse,
        corrected_test_rmse,
        test_rmse_reduction,
        relative_test_rmse_reduction,
        baseline_test_r2,
        corrected_test_r2,
        residual_corr_train,
        residual_corr_test,
        score_base,
        score_corr,
        score_base - score_corr,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 9. Generate markdown summary

This writes:

```text
docs/04_learned_residual_correction.md
```

The markdown file points to repo-local figures and summarizes the key metrics.

In [ ]:
exp.write_md(
    title="Learned Residual Correction",
    metrics_dict={
        "Baseline test RMSE": baseline_test_rmse,
        "Corrected test RMSE": corrected_test_rmse,
        "Relative test RMSE reduction": relative_test_rmse_reduction,
        "Residual correlation test": residual_corr_test,
        "Baseline residual structure score": score_base,
        "Corrected residual structure score": score_corr,
    },
    figure_names=[
        "signal_with_hidden_structure",
        "learned_residual_structure",
        "heldout_correction",
        "rmse_before_after",
        "residuals_before_after",
        "residual_spectrum_before_after",
    ],
    interpretation="""
residual → learn missing structure
learned residual → correction
coherence stabilizes without hard-coded constraint
""",
)

## 10. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
04_learned_residual_correction_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "04_learned_residual_correction_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 11. Takeaway

This notebook supports the fourth repo claim:

```text
residual structure can be learned
learned residuals can correct drift
phase-lock can stabilize coherence without hard-coded constraints
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
```